In [ ]:
# ============================================================
# CELL 1 — Installs, imports, prompt, config
# ============================================================

!pip install openai openpyxl pandas -q

import json, time
import pandas as pd
from openai import OpenAI
from google.colab import userdata

openai_client = OpenAI(api_key=userdata.get("OPENAI_API"))
OPENAI_MODEL  = "gpt-4o-mini"

SYSTEM_PROMPT = (
    "You are a fact-checking analyst. "
    "Your role is to write a concise analytical note about a political claim — "
    "surfacing what it specifically says, what context is needed to understand it, "
    "and where it may be ambiguous — without judging whether it is true or false."
)

USER_TEMPLATE = """Write a short analytical note about the political claim below.

Guidelines:
- Engage directly with the specific content of the claim: the named person, the figure cited, the policy described, the comparison drawn. Make that content the subject of your note.
- If the claim attributes a statement or action to a named person, open by naming that person and what they said or did. Do not open with a generic description of the claim.
- Where a term or figure in the claim could be read in more than one way, say what those two readings are concretely — do not say that interpretation 'depends on' something without specifying what the two outcomes would be.
- You may use contrast ('but', 'however') to show two sides of the same element.
- Do not evaluate accuracy. Banned evaluative phrasings: 'correctly states', 'misleadingly claims', 'is accurate', 'is inaccurate', 'is true', 'is false'.
- Do not use procedural language: write the relevant considerations directly, do not describe what a fact-checker 'would need to' or 'should' verify.
- Maintain a neutral, analytical tone.

Strictly forbidden — never use any of the following words or phrases:
- "ambiguity", "ambiguous", "ambiguously"
- "key ambiguity", "central ambiguity", "core ambiguity"
- "hinges on", "turns on", "centers on"
- "The assertion", "The claim asserts", "The claim presents", "The claim suggests", "The claim states", "The claim implies"
- "The statement", "This statement"
- "Relevant context"
- "Additionally", "Furthermore", "Moreover"
- "Implicit assumption"
- "Understanding this claim requires", "Requires clarity on"
- "One must consider", "It is worth considering", "It is important to note"
- "depends on how", "depends on what", "depends on whether", "depends on the"
- "would need to be defined", "would need to be understood" "rather than"

Do not follow a fixed structure. Every note must differ from the others in how it is organised. Some notes open with the speaker's name; others with the specific figure or date. Some address a single pivot in two or three sentences; others trace a narrower point in one sentence. Never apply the same sentence-by-sentence template twice. Vary how each note is organised — in the opening, in the number of sentences, and in how you close. Do not always close by generalising about interpretation.

Length: Between 30 and 90 words. Match length strictly to complexity — a single-figure claim warrants 30–45 words; a claim with multiple interacting conditions warrants 70–90 words. Do not pad.

Output: A single paragraph. No bullet points, headers, or numbered lists.

Claim: {claim}

Analytical note:"""

print("✅ Cell 1 OK")

✅ Cell 1 OK


In [ ]:
# ============================================================
# CELL 2 — Load data + build JSONL + submit batch
# ============================================================

df = pd.read_excel("claims_for_api.xlsx")
df.columns = [c.strip() for c in df.columns]

# Scegli la parte da lanciare (commenta/decommenta)
#df = df.iloc[:2100]          # Part 1                            #CHANGEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEE
# df = df.iloc[2100:4200]    # Part 2
# df = df.iloc[4200:6300]    # Part 3
# df = df.iloc[6300:8400]    # Part 4
# df = df.iloc[8400:10500]   # Part 5
df = df.iloc[10500:]       # Part 6


claim_col = "statement"
print(f"Rows loaded: {len(df)}")

# Build JSONL
jsonl_path = "openai_batch_input.jsonl"
with open(jsonl_path, "w", encoding="utf-8") as f:
    for idx, row in df.iterrows():
        claim = str(row[claim_col]).strip()
        request = {
            "custom_id": f"row-{idx}",
            "method": "POST",
            "url": "/v1/chat/completions",
            "body": {
                "model": OPENAI_MODEL,
                "max_tokens": 300,
                "temperature": 0.7,
                "messages": [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": USER_TEMPLATE.format(claim=claim)},
                ],
            },
        }
        f.write(json.dumps(request, ensure_ascii=False) + "\n")

print(f"✅ JSONL scritto: {jsonl_path}")

# Upload
with open(jsonl_path, "rb") as f:
    upload = openai_client.files.create(file=f, purpose="batch")
print(f"✅ File uploadato — ID: {upload.id}")

# Submit
batch = openai_client.batches.create(
    input_file_id=upload.id,
    endpoint="/v1/chat/completions",
    completion_window="24h",
)
OPENAI_BATCH_ID = batch.id
print(f"✅ Batch submittato — ID: {OPENAI_BATCH_ID}")
print(f"   Status: {batch.status}")

Rows loaded: 2291
✅ JSONL scritto: openai_batch_input.jsonl
✅ File uploadato — ID: file-HkpoScoxKmep1VNB4Kv5uF
✅ Batch submittato — ID: batch_6a05abdf367c81908069f35869dccf4a
   Status: validating


In [1]:
# ============================================================
# CELL 3 — Poll batch fino a completamento
# ============================================================

# Se Colab si è disconnesso, incolla qui il tuo batch ID:


from openai import OpenAI
from google.colab import userdata
import json, time
openai_client = OpenAI(api_key=userdata.get("OPENAI_API"))
OPENAI_BATCH_ID = "batch_6a05abdf367c81908069f35869dccf4a" # CHANGEEEEEEEEEEEEEEEEEEEEEEEEEEEEE !!!!!!!!!!!!!!!!!!!!!!!!!!!

print(f"Polling: {OPENAI_BATCH_ID}\n")

while True:
    batch  = openai_client.batches.retrieve(OPENAI_BATCH_ID)
    counts = batch.request_counts
    print(f"[{time.strftime('%H:%M:%S')}] {batch.status} — "
          f"{counts.completed} completed / {counts.failed} failed / {counts.total} total")

    if batch.status == "completed":
        print("\n✅ Batch completato!")
        break
    if batch.status in ("failed", "cancelled", "expired"):
        raise RuntimeError(f"Batch terminato con errore: {batch.status}\n{batch.errors}")

    time.sleep(60)

Polling: batch_6a05abdf367c81908069f35869dccf4a

[13:24:17] completed — 2291 completed / 0 failed / 2291 total

✅ Batch completato!


In [2]:
# ============================================================
# CELL 4 — Download + parse risultati
# ============================================================

raw = openai_client.files.content(batch.output_file_id).text

openai_results = {}
failed_ids     = []

for line in raw.strip().split("\n"):
    record = json.loads(line)
    idx    = int(record["custom_id"].split("-")[1])
    try:
        text = record["response"]["body"]["choices"][0]["message"]["content"].strip()
        openai_results[idx] = text
    except Exception:
        openai_results[idx] = ""
        failed_ids.append(idx)

print(f"✅ Risultati parsati: {len(openai_results)} righe")
if failed_ids:
    print(f"⚠️  Righe fallite: {failed_ids}")

✅ Risultati parsati: 2291 righe


In [3]:
# ============================================================
# CELL 5 — Salva Excel con claim + justification OpenAI
# ============================================================

from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter
import pandas as pd

df = pd.read_excel("claims_for_api.xlsx")
df.columns = [c.strip() for c in df.columns]

# Stessa parte di Cell 2
# df = df.iloc[:2100]          # Part 1           #CHANGEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEE
# df = df.iloc[2100:4200]    # Part 2
# df = df.iloc[4200:6300]    # Part 3
# df = df.iloc[6300:8400]    # Part 4
# df = df.iloc[8400:10500]   # Part 5
df = df.iloc[10500:]       # Part 6


df.columns = [c.strip() for c in df.columns]
print(df.columns.tolist())
print(df.head(2))
claim_col = "statement"
print(f"Claim column : '{claim_col}'")
print(f"Rows loaded  : {len(df)}")


df_out = df[[claim_col]].copy()
df_out.columns = ["Claim"]
df_out["OpenAI_Justification"] = [openai_results.get(i, "") for i in df_out.index]
df_out = df_out.reset_index(drop=True)

print(f"Righe totali  : {len(df_out)}")
print(f"Righe vuote   : {(df_out['OpenAI_Justification'] == '').sum()}")

output_path = "openai_just_on_original_6.xlsx"                                            #CHANGEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEE
df_out.to_excel(output_path, index=False, sheet_name="Results")

wb = load_workbook(output_path)
ws = wb["Results"]

header_fill = PatternFill("solid", fgColor="1B4F72")
header_font = Font(bold=True, color="FFFFFF", name="Arial", size=11)
cell_font   = Font(name="Arial", size=10)
wrap_align  = Alignment(wrap_text=True, vertical="top")

col_widths = [70, 80]
for col_idx, (cell, width) in enumerate(zip(ws[1], col_widths), start=1):
    cell.fill      = header_fill
    cell.font      = header_font
    cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws.column_dimensions[get_column_letter(col_idx)].width = width

ws.row_dimensions[1].height = 30
for row in ws.iter_rows(min_row=2):
    for cell in row:
        cell.font      = cell_font
        cell.alignment = wrap_align

ws.freeze_panes = "A2"
wb.save(output_path)

print(f"\n✅ Salvato: {output_path}")

# Preview
for i, row in df_out.head(3).iterrows():
    print(f"\n─── Row {i} ───")
    print(f"CLAIM:  {row['Claim'][:100]}")
    print(f"OPENAI: {row['OpenAI_Justification'][:200]}")

['statement']
                                               statement
10500  On Medicare for current retirees, hes cutting ...
10501  Sen. Marco Rubio refuses to accept the basic s...
Claim column : 'statement'
Rows loaded  : 2291
Righe totali  : 2291
Righe vuote   : 0

✅ Salvato: openai_just_on_original_6.xlsx

─── Row 0 ───
CLAIM:  On Medicare for current retirees, hes cutting $716 billion from the program.
OPENAI: The claim regarding Medicare for current retirees references a proposed reduction of $716 billion from the program. This figure can be understood as either a specific budget cut or a reduction in proj

─── Row 1 ───
CLAIM:  Sen. Marco Rubio refuses to accept the basic science on climate change and is a climate change denie
OPENAI: Sen. Marco Rubio is characterized as refusing to accept established scientific consensus on climate change and labeled a climate change denier. This description raises questions about what constitutes

─── Row 2 ───
CLAIM:  There are 10 or 20 de